# Don't Sign Anything! Rule-Based NLP Walkthrough

This notebook demonstrates how the current analysis engine works. The project does **not** use a paid LLM or trained legal model by default. It uses an explainable rule-based NLP pipeline: text cleanup, document type classification, clause pattern matching, confidence scoring, risk scoring, and plain-English report generation.

The goal is not to provide legal advice. The goal is to show potential risk signals and questions a user may want to ask before signing.

## 1. Import The Backend Analyzer

The notebook imports the same backend service used by the FastAPI `/api/analyze` endpoint.

In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

backend_path = project_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

from app.services.risk_analyzer import CLAUSE_RULES, analyze_document

len(CLAUSE_RULES)

## 2. Review The Risk Rule Library

Each `ClauseRule` has an ID, title, category, severity, regex patterns, plain-English explanation, and user-facing questions.

In [ ]:
rule_preview = [
    {
        "id": rule.id,
        "title": rule.title,
        "category": rule.category,
        "severity": rule.severity,
        "example_patterns": list(rule.patterns[:3]),
    }
    for rule in CLAUSE_RULES[:8]
]

print(json.dumps(rule_preview, indent=2))

## 3. Create Sample Agreement Text

This sample intentionally includes several common risk signals: arbitration, automatic renewal, refund restrictions, data sharing, and one-sided modification.

In [ ]:
sample_text = """
TERMS OF SERVICE

By creating an account, the user agrees to these Terms of Service. The subscription
automatically renews for successive monthly periods unless the user provides written
cancellation notice at least 30 days before the renewal date.

All fees are non-refundable and additional service charges may apply. We reserve the
right to modify these terms and fees at any time without prior notice. Continued use
of the platform means acceptance of the updated terms.

Any dispute will be resolved by binding arbitration, and the user waives any right to
participate in a class action or jury trial. We may share personal information with
affiliates, analytics providers, advertising partners, and other third parties.
"""

print(sample_text.strip())

## 4. Run Analysis

`analyze_document` returns the same structured response that the frontend dashboard receives.

In [ ]:
analysis = analyze_document(sample_text, document_name="sample_terms.txt")

summary = {
    "document_type": analysis.document_type,
    "document_type_confidence": analysis.document_type_confidence,
    "document_type_signals": analysis.document_type_signals,
    "risk_score": analysis.risk_score,
    "risk_level": analysis.risk_level,
    "finding_count": len(analysis.detected_risks),
    "word_count": analysis.word_count,
}

print(json.dumps(summary, indent=2))

## 5. Inspect Detected Risks

The analyzer shows what it found, why it matters, what exact phrase triggered the finding, and which source sentence should be reviewed.

In [ ]:
risk_rows = []
for risk in analysis.detected_risks:
    risk_rows.append(
        {
            "title": risk.title,
            "severity": risk.severity,
            "confidence": risk.confidence,
            "plain_english": risk.plain_english,
            "trigger_terms": risk.trigger_terms,
            "first_snippet": risk.matched_snippets[0] if risk.matched_snippets else "",
        }
    )

print(json.dumps(risk_rows, indent=2))

## 6. Questions And Next Steps

The app turns risk findings into simple questions a non-technical user can ask before signing.

In [ ]:
print("Questions to ask before signing:")
for question in analysis.questions_to_ask:
    print(f"- {question}")

print("\nNext steps:")
for step in analysis.next_steps:
    print(f"- {step}")

## 7. Score Explanation

The risk score is a product heuristic, not a legal rating. It combines severity, confidence, document type adjustments, user preference adjustments, and the number of findings.

In [ ]:
print(f"Risk score: {analysis.risk_score}/100")
print(f"Risk level: {analysis.risk_level}")

print("\nScoring notes:")
for note in analysis.scoring_notes:
    print(f"- {note}")

## 8. Important Limitation

This analyzer is explainable and useful for a portfolio MVP, but it is not a lawyer and does not make legal decisions. It can miss unusual wording, OCR errors, or context-specific issues. It should be presented as an educational risk assistant with a rule-based NLP engine.